In [1]:
# Download the UCI Student Performance dataset (ID 320).
student_data = fetch_ucirepo(id=320)
X_raw = student_data.data.features
y_raw = student_data.data.targets

df = pd.concat([X_raw, y_raw], axis=1)
df.columns = [str(c).strip() for c in df.columns]

print("Dataset shape:", df.shape)
display(df.head())

NameError: name 'fetch_ucirepo' is not defined

In [2]:
!pip -q install ucimlrepo

In [3]:
import os, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# Create folders where project outputs will be saved.
PROJECT_DIR = "/content/Student_Performance_Project"
GRAPH_DIR = os.path.join(PROJECT_DIR, "Graphs")
REPORT_DIR = os.path.join(PROJECT_DIR, "Reports")
os.makedirs(GRAPH_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print("Setup complete.")

Setup complete.


In [4]:
# Download the UCI Student Performance dataset (ID 320).
student_data = fetch_ucirepo(id=320)
X_raw = student_data.data.features
y_raw = student_data.data.targets

df = pd.concat([X_raw, y_raw], axis=1)
df.columns = [str(c).strip() for c in df.columns]

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (649, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [5]:
# Inspect the dataset
print("Column names:")
print(df.columns.tolist())

print("\\nData types and non-null counts:")
df.info()

print("\\nNumerical summary:")
display(df.describe(include="all").T)

print("\\nMissing values per column:")
display(df.isnull().sum().to_frame("missing_values"))

# Clean column names and remove duplicate rows if any.
df.columns = [c.strip() for c in df.columns]
df = df.drop_duplicates().copy()

# Ensure target is numeric.
df["G3"] = pd.to_numeric(df["G3"], errors="coerce")
df = df.dropna(subset=["G3"]).copy()

print("Shape after basic cleaning:", df.shape)

Column names:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']
\nData types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 649 entries, 0 to 648
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      649 non-null    object
 1   sex         649 non-null    object
 2   age         649 non-null    int64 
 3   address     649 non-null    object
 4   famsize     649 non-null    object
 5   Pstatus     649 non-null    object
 6   Medu        649 non-null    int64 
 7   Fedu        649 non-null    int64 
 8   Mjob        649 non-null    object
 9   Fjob        649 non-null    object
 10  reason      649 non-null    object

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
school,649,2,GP,423,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sex,649,2,F,383,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,649.0,NaN,NaN,NaN,16.744222,1.218138,15.0,16.0,17.0,18.0,22.0
address,649,2,U,452,NaN,NaN,NaN,NaN,NaN,NaN,NaN
famsize,649,2,GT3,457,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Pstatus,649,2,T,569,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Medu,649.0,NaN,NaN,NaN,2.514638,1.134552,0.0,2.0,2.0,4.0,4.0
Fedu,649.0,NaN,NaN,NaN,2.306626,1.099931,0.0,1.0,2.0,3.0,4.0
Mjob,649,5,other,258,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fjob,649,5,other,367,NaN,NaN,NaN,NaN,NaN,NaN,NaN


\nMissing values per column:


,missing_values
school,0
sex,0
age,0
address,0
famsize,0
Pstatus,0
Medu,0
Fedu,0
Mjob,0
Fjob,0


Shape after basic cleaning: (649, 33)


In [6]:
# NumPy practice: calculate basic statistics for final grades.
grades = df["G3"].to_numpy()
print("Mean:", np.mean(grades))
print("Median:", np.median(grades))
print("Minimum:", np.min(grades))
print("Maximum:", np.max(grades))
print("Standard deviation:", np.std(grades))

# Save a cleaned copy for the project.
df.to_csv(os.path.join(PROJECT_DIR, "student_performance_cleaned.csv"), index=False)
print("Cleaned dataset saved.")

Mean: 11.906009244992296
Median: 12.0
Minimum: 0
Maximum: 19
Standard deviation: 3.2281663339626148
Cleaned dataset saved.
